In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

# GraphConv model training

This notebook generates a training report for the GraphConv model, covering three experiments:

1. Hyperparameter optimization — selecting the best-performing model configuration.
2. Sample size optimization — quantifying the minimum number of samples needed to improve accuracy.
3. Full-sample model — training with all available data.

Results directory: `/home/mriveraceron/glv-research/tuning_results/91074c4e25b4`

Data directory: `/home/mriveraceron/glv-research/data_tensors/91074c4e25b4`

# Hyperparameter optimization

In [ ]:
# Display results table
import pandas as pd
from IPython.display import display

# Experiment directory
dir = '/home/mriveraceron/glv-research/tuning_results/91074c4e25b4'
df = pd.read_csv(f'{dir}/tuning_results.csv')
display(df.sort_values(by="pearson_corr", ascending=False))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
metrics = ['accuracy_idx', 'pearson_corr', 'spearman_corr']
titles  = ['Precisión', 'Correlación de Pearson', 'Correlación de Spearman']
lr_labels = np.sort(df['learning_rate'].unique())[::-1]  # descending
lr_map    = {lra: i for i, lra in enumerate(lr_labels)}
# Legend handles — built once, shared across all columns
combos  = df[['channels', 'layers']].drop_duplicates().reset_index(drop=True)
palette  = plt.cm.tab10.colors
offsets  = np.linspace(-0.2, 0.2, len(combos))
color_map  = {(int(r['channels']), int(r['layers'])): palette[k] for k, (_, r) in enumerate(combos.iterrows())}
offset_map = {(int(r['channels']), int(r['layers'])): offsets[k] for k, (_, r) in enumerate(combos.iterrows())}
handles = [
    mpatches.Patch(color=palette[k], label=f"n={int(r['channels'])} | c={int(r['layers'])}")
    for k, (_, r) in enumerate(combos.iterrows())
]

plt.clf()
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
for ax, metric, title, letter in zip(axes, metrics, titles, ['A', 'B', 'C']):
    for _, row in df.iterrows():
        if pd.isna(row[metric]):
            row[metric] = 0
        key   = (int(row['channels']), int(row['layers']))
        color = color_map[key]
        x     = lr_map[row['learning_rate']] + offset_map[key]
        # Plot line
        ax.plot([x, x], [0, row[metric]], color=color, lw=2, alpha=0.6)
        # Plot circle
        ax.scatter(x, row[metric], color=color, s=100, zorder=5, edgecolors='white', linewidths=0.8)
        # Value number
        ax.text(x, row[metric] + 0.01, f"{row[metric]:.2f}", ha='center', va='bottom', fontsize=7, color=color)
    # Set up x axis
    ax.set_xticks(range(len(lr_labels)))
    ax.set_xticklabels([f"{lra:.0e}" for lra in lr_labels], fontsize=9)
    ax.set_xlabel('Tasa de aprendizaje', fontsize=11, labelpad=10)
    ax.set_ylim(-0.25, df[metric].max() * 1.15)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    # Remove top and right border
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Add legend at top right
    ax.legend(handles=handles, title='Neuronas | Capas', loc='upper right',
              frameon=True, framealpha=0.4, fontsize=9, title_fontsize=10)
    ax.text(-0.05, 1.05, letter, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='bottom')


plt.suptitle('Desempeño del modelo por tasa de aprendizaje', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])  # leaves 5% headroom at the top for the suptitle
plt.tight_layout()
plt.savefig('/home/mriveraceron/glv-research/tuning_results/91074c4e25b4/variants_panel.png')
plt.show()

## Caption
Figure 1. Model performance across hyperparameter configurations (channels, layers, and learning rate). (A) Positive predictive value, (B) Pearson correlation coefficient, and (C) Spearman correlation coefficient.

## Interpretation
Based on Figure 1, the optimal hyperparameter configuration corresponds to a learning rate of 1e-3 and 64 channels, with either 5 or 10 layers yielding comparable performance.

The selected model (64 channels, 5 layers, lr = 1e-3) was chosen on the basis of its Spearman 
correlation, where it outperforms all other configurations. Although this model does not achieve the highest precision, halving the number of layers relative to the 10-layer counterpart reduces computational cost without a meaningful loss in predictive performance.

# Sample size optimization

Results directory: `/home/mriveraceron/glv-research/tuning_results/GraphConv_ss`

Data directory: `/home/mriveraceron/glv-research/data_null`


In [ ]:
# Display results table
import pandas as pd
from IPython.display import display

# Experiment directory
dir = '/home/mriveraceron/glv-research/tuning_results/GraphConv_ss'
df = pd.read_csv(f'{dir}/tuning_results_V2.csv')
display(df.sort_values(by="pearson_corr", ascending=False))

In [ ]:
# Elapsed time plot
import matplotlib.pyplot as plt
import numpy as np


plt.clf()
plt.close('all')
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=False)

#------------------------- Elapsed time plot -------------------
